---
## ***Notebook 2: Data Cleaning & Preprocessing***
---

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

In [2]:
#Load Dataset
df = pd.read_csv("naukri_jobs.csv")

In [3]:
# Overview
print("="*50)
print("Dataset Shape")
print("="*50)

print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nFirst Five Rows")
display(df.head())

Dataset Shape
(4136, 12)

Columns
['Role Searched', 'Job Title', 'Company', 'Rating', 'Reviews', 'Experience', 'Salary', 'Location', 'Description', 'Skills', 'Posted', 'Job Link']

First Five Rows


,Role Searched,Job Title,Company,Rating,Reviews,Experience,Salary,Location,Description,Skills,Posted,Job Link
0,Data Analyst,Climate Data Analyst,MSCI Services,3.6,392 Reviews,0-4 Yrs,Not Disclosed,Pune,The Climate Data and Content team is responsib...,"data operations, data analysis, data analytics...",3 weeks ago,https://www.naukri.com/job-listings-climate-da...
1,Data Analyst,Data Analyst,Snapdeal,3.6,673 Reviews,1-6 Yrs,Not Disclosed,Gurugram,Providing technical expertise in data storage ...,"master data, python, metadata, data analysis, ...",3+ weeks ago,https://www.naukri.com/job-listings-data-analy...
2,Data Analyst,Data Analyst,Cognizant,3.7,63144 Reviews,5-10 Yrs,Not Disclosed,Hyderabad,Role: Data AnalystExperience: 5+ years Preferr...,"Data Analysis, Data Visualization, Advanced Ex...",1 week ago,https://www.naukri.com/job-listings-data-analy...
3,Data Analyst,Data Analyst,MNC,NaN,NaN,0-2 Yrs,2.25-3.5 Lacs PA,"Pimpri-Chinchwad, Pune",We are looking for a motivated and detail-orie...,"Google Sheets, Power BI, MS Office, Data Analy...",2 days ago,https://www.naukri.com/job-listings-data-analy...
4,Data Analyst,Data Analyst (English Required),Peroptyx,3.2,17 Reviews,0-5 Yrs,Not Disclosed,Remote,"For thousands of years, maps have provided hum...","Data Analysis, English, Data Analytics, Data M...",2 weeks ago,https://www.naukri.com/job-listings-data-analy...


In [4]:
# Dataset Information
print("="*50)
print("Dataset Information")
print("="*50)

df.info()

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4136 entries, 0 to 4135
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Role Searched  4136 non-null   object 
 1   Job Title      4136 non-null   object 
 2   Company        4136 non-null   object 
 3   Rating         3177 non-null   float64
 4   Reviews        3177 non-null   object 
 5   Experience     4084 non-null   object 
 6   Salary         4136 non-null   object 
 7   Location       4114 non-null   object 
 8   Description    4106 non-null   object 
 9   Skills         4103 non-null   object 
 10  Posted         4136 non-null   object 
 11  Job Link       4136 non-null   object 
dtypes: float64(1), object(11)
memory usage: 387.9+ KB


In [5]:
# Missing Value
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(df.isnull().mean()*100,2)
})

missing.sort_values("Missing Values", ascending=False)

,Missing Values,Percentage
Rating,959,23.19
Reviews,959,23.19
Experience,52,1.26
Skills,33,0.80
Description,30,0.73
Location,22,0.53
Role Searched,0,0.00
Job Title,0,0.00
Company,0,0.00
Salary,0,0.00


In [6]:
# Duplicate Records
duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)

df = df.drop_duplicates()

df.reset_index(drop=True, inplace=True)

print(df.shape)
print("Duplicate Records after removal:", duplicates)

Duplicate Records: 0
(4136, 12)
Duplicate Records after removal: 0


In [7]:
# DataTypes
df.dtypes

Role Searched     object
Job Title         object
Company           object
Rating           float64
Reviews           object
Experience        object
Salary            object
Location          object
Description       object
Skills            object
Posted            object
Job Link          object
dtype: object

In [8]:
#Cleaning Experience column
# Minimum Experience
df["Min Experience"] = (
    df["Experience"]
    .str.extract(r'(\d+)')
    .astype(float)
)
# Maximum Experience
df["Max Experience"] = (
    df["Experience"]
    .str.extract(r'-(\d+)')
    .astype(float)
)
# Average Experience
df["Average Experience"] = (
    df["Min Experience"] +
    df["Max Experience"]
)/2

In [9]:
# Clean Rating
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

In [10]:
# Clean Reviews
df["Reviews"] = (
    df["Reviews"]
    .str.replace("Reviews","", regex=False)
    .str.replace(",","")
    .str.strip()
)

df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")

In [11]:
# Clean Salary
# Minimum Salary
df["Min Salary"] = (
    df["Salary"]
    .str.extract(r'(\d+\.?\d*)')
)
# Maximum Salary
df["Max Salary"] = (
    df["Salary"]
    .str.extract(r'-(\d+\.?\d*)')
)
# Convert DataTypes
df["Min Salary"] = pd.to_numeric(df["Min Salary"], errors="coerce")

df["Max Salary"] = pd.to_numeric(df["Max Salary"], errors="coerce")
# Average Salary
df["Average Salary"] = (
    df["Min Salary"] +
    df["Max Salary"]
)/2

In [26]:
# Clean Location
df["Primary Location"] = df["Location"].copy()
df["Primary Location"] = (
    df["Primary Location"]
        .str.split(",")
        .str[0]
        .str.replace("Hybrid - ", "", regex=False)
        .str.replace("Remote - ", "", regex=False)
        .str.strip()
)

In [27]:
# Clean Skills
# Lowercase
df["Skills"] = df["Skills"].str.lower()
# Remove Duplicates
df["Skills"] = (
    df["Skills"]
    .str.replace(",",",",regex=False)
)

In [28]:
# Converting posted into date format
import re
def convert_posted(posted):
    if pd.isna(posted):
        return np.nan
    posted = posted.lower().strip()
    if posted in ["today", "just now"]:
        return 0
    number = re.search(r"\d+", posted)
    if number:
        number = int(number.group())
        if "day" in posted:
            return number
        elif "week" in posted:
            return number * 7
        elif "month" in posted:
            return number * 30
        elif "year" in posted:
            return number * 365
    return np.nan

In [29]:
df["Posted Days"] = df["Posted"].apply(convert_posted)

In [42]:
# FEATURE ENGINEERING
# Experience Level
def experience_level(x):

    if pd.isna(x):
        return np.nan
    elif x <= 2:
        return "Entry"
    elif x <= 5:
        return "Junior"
    elif x <= 8:
        return "Mid"
    else:
        return "Senior"

In [43]:
df["Experience Level"] = df["Average Experience"].apply(experience_level)

In [44]:
# Salary Category
def salary_category(x):
    if pd.isna(x):
        return np.nan
    elif x < 5:
        return "Low"
    elif x < 10:
        return "Medium"
    elif x < 20:
        return "High"
    else:
        return "Very High"

In [45]:
df["Salary Category"] = df["Average Salary"].apply(salary_category)

In [46]:
# Number Of Skills
df["Skill Count"] = df["Skills"].str.count(",") + 1

In [51]:
df["Skills"] = (
    df["Skills"]
      .str.replace("artificial intelligence", "ai", regex=False)
      .str.replace("data analytics", "data analysis", regex=False)
      .str.replace("analytical", "analytics", regex=False)
)

In [52]:
df["Company"] = (
    df["Company"]
      .str.replace("Leading Client", "Confidential", regex=False)
)

In [53]:
# Final Check
print(df.info())

display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4136 entries, 0 to 4135
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Role Searched       4136 non-null   object 
 1   Job Title           4136 non-null   object 
 2   Company             4136 non-null   object 
 3   Rating              3177 non-null   float64
 4   Reviews             3177 non-null   float64
 5   Experience          4084 non-null   object 
 6   Salary              4136 non-null   object 
 7   Location            4114 non-null   object 
 8   Description         4106 non-null   object 
 9   Skills              4103 non-null   object 
 10  Posted              4136 non-null   object 
 11  Job Link            4136 non-null   object 
 12  Min Experience      4084 non-null   float64
 13  Max Experience      4078 non-null   float64
 14  Average Experience  4078 non-null   float64
 15  Min Salary          470 non-null    float64
 16  Max Sa

,Role Searched,Job Title,Company,Rating,Reviews,Experience,Salary,Location,Description,Skills,...,Max Experience,Average Experience,Min Salary,Max Salary,Average Salary,Posted Days,Experience Level,Salary Category,Skill Count,Primary Location
0,Data Analyst,Climate Data Analyst,MSCI Services,3.6,392.0,0-4 Yrs,Not Disclosed,Pune,The Climate Data and Content team is responsib...,"data operations, data analysis, data analysis,...",...,4.0,2.0,NaN,NaN,NaN,21.0,Entry,NaN,8.0,Pune
1,Data Analyst,Data Analyst,Snapdeal,3.6,673.0,1-6 Yrs,Not Disclosed,Gurugram,Providing technical expertise in data storage ...,"master data, python, metadata, data analysis, ...",...,6.0,3.5,NaN,NaN,NaN,21.0,Junior,NaN,8.0,Gurugram
2,Data Analyst,Data Analyst,Cognizant,3.7,63144.0,5-10 Yrs,Not Disclosed,Hyderabad,Role: Data AnalystExperience: 5+ years Preferr...,"data analysis, data visualization, advanced ex...",...,10.0,7.5,NaN,NaN,NaN,7.0,Mid,NaN,8.0,Hyderabad
3,Data Analyst,Data Analyst,MNC,NaN,NaN,0-2 Yrs,2.25-3.5 Lacs PA,"Pimpri-Chinchwad, Pune",We are looking for a motivated and detail-orie...,"google sheets, power bi, ms office, data analy...",...,2.0,1.0,2.25,3.5,2.875,2.0,Entry,Low,8.0,Pimpri-Chinchwad
4,Data Analyst,Data Analyst (English Required),Peroptyx,3.2,17.0,0-5 Yrs,Not Disclosed,Remote,"For thousands of years, maps have provided hum...","data analysis, english, data analysis, data ma...",...,5.0,2.5,NaN,NaN,NaN,14.0,Junior,NaN,8.0,Remote


In [54]:
df.to_csv("cleaned_naukri_jobs.csv", index=False)
print("Dataset Saved Successfully")

Dataset Saved Successfully


In [55]:
# Missing Value
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(df.isnull().mean()*100,2)
})

missing.sort_values("Missing Values", ascending=False)

,Missing Values,Percentage
Max Salary,3673,88.81
Salary Category,3673,88.81
Average Salary,3673,88.81
Min Salary,3666,88.64
Rating,959,23.19
Reviews,959,23.19
Max Experience,58,1.40
Experience Level,58,1.40
Average Experience,58,1.40
Experience,52,1.26


### **Explanation -**
The second notebook focused on transforming the raw scraped dataset into a clean and analysis-ready dataset. Since web-scraped data often contains missing values, inconsistent formats, and text-based fields, I first inspected the dataset using shape, data types, and missing value analysis to understand its quality. Duplicate records were removed to eliminate repeated job postings while preserving genuine listings.

Several columns required preprocessing before they could be used for analysis or machine learning. For example, the Experience column originally contained ranges such as "0–4 Yrs," which were split into minimum, maximum, and average experience values to create meaningful numerical features. Similarly, salary values were separated into minimum, maximum, and average salary wherever salary information was available. Jobs with "Not Disclosed" salaries were intentionally left as missing values rather than assigning arbitrary numbers, as this accurately reflects real-world employer behavior and avoids introducing bias into the dataset.

Text-based fields such as reviews, skills, and locations were standardized by removing unnecessary text and formatting inconsistencies. I also engineered new features including Experience Level, Salary Category, Skill Count, and Posted Days. These derived features improve interpretability, enable more meaningful visualizations, and provide useful inputs for machine learning models. Rather than filling every missing value, I preserved missing information where it represented genuine business scenarios, such as companies without ratings or undisclosed salaries. The final output was a clean, structured dataset suitable for exploratory data analysis, predictive modeling, and dashboard creation.